[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ChrisW09/Python-for-AI-Driven-Automation/blob/main/02_data_science/09_matplotlib_basics.ipynb)

# 📓 Notebook 9 — Visualization Basics: Great Charts in a Few Lines

> **Module:** Data Science Libraries · **Estimated time:** 35–50 min · **Difficulty:** Beginner

A plot answers questions that tables cannot. Distributions, trends, outliers, relationships — they are *seen* before they are *understood*. The good news: you do **not** need twenty lines of matplotlib boilerplate to get there. Modern Python gives you excellent charts in **one to three lines**:

- **pandas `.plot()`** — draw straight from the DataFrame you already have,
- **seaborn** — beautiful statistical charts with smart defaults,
- **matplotlib** — the engine underneath both; you call it directly only for quick array plots and final tweaks.

Every example is anchored in **AI support-operations data**: cost per call, latency distributions, automation rates, model comparisons. By the end you will have built the **2×2 executive dashboard** used in the NB 24 capstone — the easy way.

## 🎯 Learning objectives

1. Draw **line, bar, scatter, histogram, and box plots in 1–3 lines** of code.
2. Plot directly from a DataFrame or Series with **`.plot()`**.
3. Split any chart by category with seaborn's **`hue=`** — one argument instead of a loop.
4. Build a **correlation heatmap** and a **pairplot** as one-liners.
5. Compose a **2×2 dashboard** by passing `ax=` to pandas/seaborn.
6. Add just enough polish (title, labels, size) and **save figures to PNG**.

## ✅ Prerequisites

Notebooks 1–8 (pandas in particular).

## 1. Setup — three imports and one style line

We use three libraries together, and they are friends, not rivals:

- **pandas** — you already have your data in a DataFrame; `.plot()` draws it directly.
- **seaborn** — statistical charts with beautiful defaults; imported as `sns`.
- **matplotlib** — the engine underneath both; we call it directly only for small touch-ups.

`sns.set_theme()` restyles *every* chart in the notebook — including pandas and raw matplotlib plots — because everything renders through matplotlib.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")   # one line → every plot below looks polished

print(f"pandas {pd.__version__} · seaborn {sns.__version__}")

> 💡 **One styling line instead of a dozen.** The old-school way is a `plt.rcParams.update({...})` block with ten hand-picked settings. `sns.set_theme()` does that for you. Try `style="darkgrid"`, `"white"`, or `"ticks"` if you prefer a different look.
>
> 💡 **The trailing semicolon.** When a plotting call is the last line of a cell, Jupyter echoes its return value (something like `<Axes: >`) above the chart. End the line with `;` to suppress that noise.

## 2. The dataset — 500 LLM support calls

One realistic DataFrame carries the whole notebook: 500 LLM calls from an AI support operation, with the model used, the support channel, token count, cost, latency, and a customer-satisfaction score.

In [ ]:
rng = np.random.default_rng(42)
n = 500

calls = pd.DataFrame({
    "model":     rng.choice(["gpt-4o-mini", "claude-haiku"], n),
    "channel":   rng.choice(["Email", "Chat", "Phone", "Web Form"], n, p=[0.3, 0.4, 0.2, 0.1]),
    "tokens_in": rng.integers(120, 2_000, n),
})
calls["cost_usd"]     = calls["tokens_in"] / 1000 * np.where(calls["model"] == "gpt-4o-mini", 0.0006, 0.0008)
calls["latency_ms"]   = np.where(calls["model"] == "gpt-4o-mini",
                                 rng.normal(1800, 400, n), rng.normal(2300, 500, n)).clip(min=200)
calls["satisfaction"] = (4.6 - 600 * calls["cost_usd"] + rng.normal(0, 0.4, n)).clip(1, 5)

calls.head()

…and one small monthly table for trends: the share of conversations resolved without a human, per channel.

In [ ]:
monthly = pd.DataFrame({
    "Chat":     [0.55, 0.58, 0.60, 0.63, 0.65, 0.68, 0.70, 0.72, 0.74, 0.76, 0.78, 0.81],
    "Email":    [0.42, 0.45, 0.47, 0.50, 0.52, 0.55, 0.57, 0.60, 0.62, 0.65, 0.67, 0.70],
    "Phone":    [0.15, 0.16, 0.17, 0.18, 0.20, 0.21, 0.22, 0.23, 0.24, 0.25, 0.26, 0.28],
    "Web Form": [0.62, 0.65, 0.68, 0.70, 0.72, 0.74, 0.76, 0.78, 0.80, 0.82, 0.83, 0.85],
}, index=pd.Index(range(1, 13), name="month"))

monthly.head(3)

## 3. Line plots — a trend in one line

A DataFrame knows how to draw itself. `.plot()` uses the index as the x-axis, draws every column as its own line, and builds the legend from the column names:

In [ ]:
monthly.plot(title="Automation rate by channel", ylabel="automation rate", ylim=(0, 1));

One line — and it replaced about fifteen lines of classic matplotlib (four `ax.plot(...)` calls, manual colors, a legend, labels, limits). Everything came from the DataFrame itself: x-axis from the index, one line per column, legend from the column names.

A single column works the same way:

```python
monthly["Chat"].plot(title="Automation rate — Chat");
```

And when you have plain **arrays or lists** instead of a DataFrame, raw matplotlib is still the quickest two-liner:

```python
plt.plot(range(1, 13), rates, marker="o")
plt.title("Automation rate — Chat");
```

## 4. Bar charts — comparing categories

Comparing categories almost always starts with a `groupby` — and the result plots itself:

In [ ]:
calls.groupby("channel")["cost_usd"].sum().plot(kind="bar", title="Total spend by channel", ylabel="USD", rot=0);

Two habits that make bar charts instantly more readable:

1. **Sort first** — `.sort_values()` turns an arbitrary order into a ranking the eye can follow.
2. **Go horizontal for long labels** — `kind="barh"` keeps category names readable instead of rotating them.

In [ ]:
calls["channel"].value_counts().sort_values().plot(kind="barh", title="Calls per channel");

When the bar height is a **mean**, use seaborn — `sns.barplot` computes the group means *and* adds 95% confidence intervals automatically, so the viewer can see which differences are solid and which are noise:

In [ ]:
sns.barplot(data=calls, x="channel", y="satisfaction")
plt.title("Mean satisfaction per channel (error bars = 95% CI)");

## 5. Histograms — distributions

A histogram tells you the *shape* of one variable: bell-shaped, skewed, bimodal? Crucial before you start modelling — and one line away:

In [ ]:
calls["latency_ms"].plot(kind="hist", bins=30, title="Latency distribution", xlabel="latency (ms)");

The next question is almost always *“does the distribution differ by group?”* In classic matplotlib that means two `hist` calls, manual `alpha`, manual colors, a manual legend. In seaborn it is **one argument**: `hue=`.

In [ ]:
sns.histplot(data=calls, x="latency_ms", hue="model", bins=30)
plt.title("Latency by model");

> 💡 **`hue=` is seaborn's superpower.** Almost every seaborn function accepts it, and it always means the same thing: *split this chart by category — colors, legend and all.* Remember this one argument and you can retire a whole class of for-loops.
>
> 💡 Add `kde=True` to overlay a smooth density curve, or use `sns.kdeplot(...)` for the curves alone.

## 6. Scatter plots — relationships

For *“is X related to Y?”* use a scatter. pandas has it built in:

In [ ]:
calls.plot.scatter(x="cost_usd", y="satisfaction", alpha=0.5, title="Cost vs satisfaction");

Color the points by category? Same trick as before — `hue=`:

In [ ]:
sns.scatterplot(data=calls, x="cost_usd", y="satisfaction", hue="model", alpha=0.6)
plt.title("Cost vs satisfaction, by model");

And the trend line — which used to mean `np.polyfit` plus a manual line plot — is one call with `sns.regplot`:

In [ ]:
sns.regplot(data=calls, x="cost_usd", y="satisfaction",
            scatter_kws={"alpha": 0.3}, line_kws={"color": "crimson"})
plt.title("Cost vs satisfaction — with trend line");

**Reading the chart.** The downward trend line says higher-cost calls tend to come with slightly lower satisfaction — plausibly because expensive calls are the hard ones the bot couldn't fully handle. The shaded band around the line is a confidence interval, which `regplot` adds for free.

## 7. Box plots — several distributions side by side

When you want to compare *several* distributions at once, a box plot is denser than a row of histograms — and `sns.boxplot` needs only the column names:

In [ ]:
sns.boxplot(data=calls, x="model", y="latency_ms")
plt.title("Latency per model");

**Reading a box plot.** The box covers the interquartile range (Q1 to Q3), the line inside is the **median**, whiskers extend to data within 1.5×IQR, and points beyond are outliers. A wide box = high variance; a tight box = consistent performance.

> 💡 Two free upgrades, both still one line: add `hue="channel"` for grouped boxes, or swap in `sns.violinplot(...)` to see the full shape of each distribution instead of just its quartiles.

## 8. Heatmaps — a matrix in one line

Heatmaps are perfect for correlation matrices, confusion matrices, or any “every-row-by-every-column” comparison. `annot=True` writes the numbers into the cells — something that takes a nested for-loop in raw matplotlib:

In [ ]:
num_cols = ["tokens_in", "cost_usd", "latency_ms", "satisfaction"]

sns.heatmap(calls[num_cols].corr(), annot=True, fmt=".2f", cmap="Blues", vmin=-1, vmax=1)
plt.title("Correlation matrix");

**Reading the heatmap.** `tokens_in` and `cost_usd` are almost perfectly correlated (cost is driven by tokens), cost and satisfaction are negatively related, and latency is driven by the *model*, not by the numeric columns — which is why its correlations are weak.

**The same one-liner draws confusion matrices.** In NB 14 you will evaluate a real classifier — `sns.heatmap(cm, annot=True)` is all it takes to visualize where the model is confused.

## 9. Many charts at once

### The ultimate one-liner: `sns.pairplot`

One call draws a scatter plot for every pair of numeric columns plus a distribution on the diagonal — colored by category. It is the fastest “show me everything” command in Python:

In [ ]:
sns.pairplot(calls[["tokens_in", "cost_usd", "latency_ms", "satisfaction", "model"]],
             hue="model", height=2);

### A 2×2 dashboard — one layout line, four one-liners

Sometimes you want a *specific* set of panels. Create the grid with a single `plt.subplots(2, 2)` call, then fill each panel by passing `ax=` to pandas or seaborn — every plotting tool in this notebook accepts it. This is exactly the layout of the **NB 24 capstone** — and you just learned the short way to build it.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(11, 7))
fig.suptitle("AI-ops dashboard — 500-call sample", fontweight="bold")

calls["tokens_in"].plot(kind="hist", bins=25, ax=axes[0, 0], title="Tokens per call")
sns.scatterplot(data=calls, x="latency_ms", y="cost_usd", hue="model", s=15, ax=axes[0, 1])
calls.groupby("model")["cost_usd"].mean().plot(kind="bar", ax=axes[1, 0], title="Mean cost per call", rot=0)
sns.boxplot(data=calls, x="model", y="latency_ms", ax=axes[1, 1])

axes[0, 1].set_title("Cost vs latency")
axes[1, 1].set_title("Latency per model")
fig.tight_layout();

## 10. Polish & saving — just enough matplotlib

Most polish is an **argument**, not extra code: `title=`, `xlabel=`, `ylabel=`, `figsize=`, `ylim=`, `rot=`, `legend=`. When you need more, remember that every pandas and seaborn call **returns a matplotlib `Axes`** — so the full matplotlib toolbox is one variable away:

In [ ]:
ax = monthly.plot(figsize=(8, 4), title="Automation rate by channel", ylabel="automation rate")
ax.set_ylim(0, 1)                  # any matplotlib method works on the returned Axes
ax.legend(loc="lower right")

# plt.savefig("automation_rate.png", dpi=200, bbox_inches="tight")   # ← uncomment to save

> 💡 **Saving figures.** Call `plt.savefig("name.png", dpi=200, bbox_inches="tight")` in the same cell as the plot. Use `.png` for reports and the web, `.pdf`/`.svg` for slides and papers (infinitely scalable vectors).
>
> 💡 **Annotations.** Need an arrow pointing at the punchline? That's the one job where raw matplotlib is still the tool: `ax.annotate("peak", xy=(x, y), xytext=(x2, y2), arrowprops=dict(arrowstyle="->"))`.

## 11. Which tool when — and the classic pitfalls

| You have… | Reach for | Example |
|-----------|-----------|---------|
| a DataFrame/Series, want a quick look | **pandas `.plot()`** | `df.plot()`, `s.plot(kind="barh")` |
| a comparison across categories / a statistical chart | **seaborn** | `sns.boxplot(data=df, x="g", y="v", hue="h")` |
| plain arrays, or pixel-level control | **matplotlib** | `plt.plot(x, y)`, `ax.annotate(...)` |
| a matrix | **seaborn heatmap** | `sns.heatmap(m, annot=True)` |

| Pitfall | Symptom | Fix |
|---------|---------|-----|
| Forgot `kind="bar"` on a category Series | meaningless zigzag line | `.plot(kind="bar")` |
| `<Axes: …>` text above every chart | visual noise | end the plotting line with `;` |
| Long category labels overlap | unreadable x-axis | `kind="barh"`, or `rot=45` |
| Unsorted bars | ranking is hard to read | `.sort_values()` before `.plot` |
| No title / axis labels | chart is a riddle | `title=`, `xlabel=`, `ylabel=` arguments |
| Pie charts | proportions are hard to compare | use a (sorted) bar chart |

## 🧪 Practice exercises

### Exercise 1 — ⭐ One-line line plot

Plot `quarterly_revenue` (defined in the cell below) as a line with circle markers, a title, and a y-axis label — in a **single statement**.

In [ ]:
# Your code here  👇
quarterly_revenue = pd.Series([210, 235, 260, 305], index=["Q1", "Q2", "Q3", "Q4"], name="revenue")


<details>
<summary>💡 <b>Solution</b></summary>

```python
quarterly_revenue.plot(marker="o", title="Revenue by quarter", ylabel="revenue (k$)");
```

Index → x-axis, values → y-axis, everything else is an argument. This is the default move whenever the data is already a Series or DataFrame.
</details>

### Exercise 2 — ⭐ Sorted horizontal bars

In one statement: compute the **mean satisfaction per channel** from `calls`, sort it, and draw it as a horizontal bar chart with a title.

In [ ]:
# Your code here  👇


<details>
<summary>💡 <b>Solution</b></summary>

```python
calls.groupby("channel")["satisfaction"].mean().sort_values().plot(
    kind="barh", title="Mean satisfaction by channel", xlabel="satisfaction (1–5)");
```

`groupby → aggregate → sort → plot` is the single most-used chart pipeline in business analytics — worth committing to muscle memory.
</details>

### Exercise 3 — ⭐⭐ Distributions split by category

Show the distribution of `cost_usd` split by `model` in one seaborn call, with a smooth density curve on top (hint: `kde=True`). Which model is more expensive per call?

In [ ]:
# Your code here  👇


<details>
<summary>💡 <b>Solution</b></summary>

```python
sns.histplot(data=calls, x="cost_usd", hue="model", bins=30, kde=True)
plt.title("Cost per call, by model");
```

`claude-haiku` sits visibly to the right — it costs more per token in this synthetic pricing, and the chart shows it without a single number being printed.
</details>

### Exercise 4 — ⭐⭐ Scatter with a trend line

Use `sns.regplot` to check whether `tokens_in` predicts `latency_ms` in `calls`. Make the points translucent so over-plotting doesn't hide the pattern. What does the trend line tell you?

In [ ]:
# Your code here  👇


<details>
<summary>💡 <b>Solution</b></summary>

```python
sns.regplot(data=calls, x="tokens_in", y="latency_ms",
            scatter_kws={"alpha": 0.3}, line_kws={"color": "crimson"})
plt.title("Tokens vs latency");
```

The trend line is essentially **flat**: in this dataset latency is driven by *which model* answered, not by how long the prompt was. A flat regression line is a finding too — “no relationship” is often the most valuable thing a scatter can tell you.
</details>

### Exercise 5 — ⭐⭐ Refactor me 🐞

The cell below works, but it is doing everything by hand: a loop over the categories, manual colors, manual labels, a manual legend. Rewrite it in **two lines** using what you learned in this notebook.

In [ ]:
# 👇 This works — now rewrite it in 2 lines below it.
fig, ax = plt.subplots(figsize=(8, 5))
for m, colour in [("gpt-4o-mini", "#4C72B0"), ("claude-haiku", "#DD8452")]:
    sub = calls[calls["model"] == m]
    ax.scatter(sub["latency_ms"], sub["cost_usd"], s=18, alpha=0.6, c=colour, label=m)
ax.set_xlabel("latency_ms")
ax.set_ylabel("cost_usd")
ax.set_title("Cost vs latency, by model")
ax.legend(title="model")
plt.show()


<details>
<summary>💡 <b>Solution</b></summary>

```python
sns.scatterplot(data=calls, x="latency_ms", y="cost_usd", hue="model", s=18, alpha=0.6)
plt.title("Cost vs latency, by model");
```

Ten lines → two. `hue="model"` replaces the loop, the colors, *and* the legend; the axis labels come from the column names for free. Whenever you catch yourself writing a for-loop around `ax.scatter`/`ax.hist`, there is almost always a `hue=` waiting to replace it.
</details>

## 🧠 Stretch exercises

A few more applied exercises to deepen the material. Try them yourself before opening the solutions.

### Stretch exercise A — ⭐⭐⭐ The everything-plot

Run `sns.pairplot` on the numeric columns of `calls`, colored by `model`, and write down **two patterns** you can read off it. Which panels repeat what the correlation heatmap in section 8 already told you?

In [ ]:
# Your code here  👇


<details>
<summary>💡 <b>Solution</b></summary>

```python
sns.pairplot(calls[["tokens_in", "cost_usd", "latency_ms", "satisfaction", "model"]],
             hue="model", height=2);
```

Patterns to spot: (1) `tokens_in` vs `cost_usd` is two perfect lines — one per model, because each model has a fixed price per token; (2) the `latency_ms` diagonal shows two shifted distributions — the models differ in speed; (3) `satisfaction` declines as `cost_usd` rises. The pairplot is the heatmap's “show me, don't tell me” twin: same information, but you also see *shapes* and *outliers*, not just one correlation number per pair.
</details>

### Stretch exercise B — ⭐⭐⭐ Small multiples in one line

Show `cost_usd` vs `satisfaction` as **four small scatter panels, one per channel**, colored by model — all in a single seaborn call.

Hint: `sns.relplot(..., col="channel", col_wrap=2)`.

In [ ]:
# Your code here  👇


<details>
<summary>💡 <b>Solution</b></summary>

```python
sns.relplot(data=calls, x="cost_usd", y="satisfaction",
            col="channel", col_wrap=2, hue="model", height=3);
```

`col=` is `hue=`'s big sibling: instead of splitting by *color*, it splits by *panel*. One line gives you a grid of comparable charts — the “small multiples” pattern that data-visualization legend Edward Tufte called the most underused design in analytics.
</details>

### Stretch exercise C — ⭐⭐⭐ Two stacked panels, shared x-axis

Plot the Chat automation rate (top panel) and its **month-over-month change** (bottom panel) as two stacked charts sharing the x-axis. Hint: `plt.subplots(2, 1, sharex=True)` for the grid, `.diff()` for the change, and `ax=` to aim each pandas plot at its panel.

In [ ]:
# Your code here  👇


<details>
<summary>💡 <b>Solution</b></summary>

```python
fig, (ax1, ax2) = plt.subplots(2, 1, sharex=True, figsize=(8, 5))
monthly["Chat"].plot(ax=ax1, marker="o", title="Chat automation rate")
monthly["Chat"].diff().plot(ax=ax2, marker="o", title="Month-over-month change")
fig.tight_layout();
```

Four lines for a two-panel figure: one line builds the grid, one pandas one-liner fills each panel, and `sharex=True` keeps the months aligned so the reader's eye can travel vertically between the level and its change.
</details>

### Stretch exercise D — ⭐⭐⭐ Grouped box plots

Is the latency gap between the two models consistent across channels? Answer with **one chart in one line**: latency per channel, split by model.

Hint: `x=`, `y=`, and `hue=` on `sns.boxplot`.

In [ ]:
# Your code here  👇


<details>
<summary>💡 <b>Solution</b></summary>

```python
sns.boxplot(data=calls, x="channel", y="latency_ms", hue="model")
plt.title("Latency by channel and model");
```

Eight box plots, automatically grouped, colored, and legended — from one line. The gap between the two models is roughly the same in every channel, which tells you the latency difference is a *model* property, not a *channel* property. That's a two-variable interaction question answered without writing a single loop.
</details>

## 🎁 Bonus mini-project — Build a 2×2 AI-ops dashboard

The cell below generates a *fresh* dataset of 500 LLM calls. Build a 2×2 dashboard for it:

1. **(0,0)** Histogram of `tokens_in`.
2. **(0,1)** Scatter of `latency_ms` vs `cost_usd`, colored by `model`.
3. **(1,0)** Bar chart of mean `cost_usd` per `model`.
4. **(1,1)** Box plot of `latency_ms` per `model`.

One layout line + four one-liners is all it takes. This is structurally *the same dashboard* used in the NB 24 capstone — master the shape and you can produce a polished one-pager for any analysis.

In [ ]:
# Your code here  👇
rng = np.random.default_rng(0)
n = 500

df = pd.DataFrame({
    "tokens_in": rng.integers(120, 2_000, n),
    "model":     rng.choice(["gpt-4o-mini", "claude-haiku"], n),
})
df["cost_usd"]   = df["tokens_in"] / 1000 * np.where(df["model"] == "gpt-4o-mini", 0.0006, 0.0008)
df["latency_ms"] = np.where(df["model"] == "gpt-4o-mini",
                            rng.normal(1800, 400, n), rng.normal(2300, 500, n)).clip(min=200)


<details>
<summary>💡 <b>Solution</b></summary>

```python
fig, axes = plt.subplots(2, 2, figsize=(11, 7))
fig.suptitle("AI-ops dashboard — 500-call sample", fontweight="bold")

df["tokens_in"].plot(kind="hist", bins=25, ax=axes[0, 0], title="Tokens per call")
sns.scatterplot(data=df, x="latency_ms", y="cost_usd", hue="model", s=15, ax=axes[0, 1])
df.groupby("model")["cost_usd"].mean().plot(kind="bar", ax=axes[1, 0], title="Mean cost per call", rot=0)
sns.boxplot(data=df, x="model", y="latency_ms", ax=axes[1, 1])

axes[0, 1].set_title("Cost vs latency")
axes[1, 1].set_title("Latency per model")
fig.tight_layout();
```

**Why this matters.** You just produced the exact deliverable an engineering manager would paste into a model-selection slide — in under ten lines. The NB 24 capstone uses the same skeleton.
</details>

## 🧠 Key takeaways

1. **Reach for the one-liner first.** `df.plot(...)` for quick looks, seaborn for statistical charts, raw matplotlib for plain arrays and final tweaks.
2. `groupby(...).agg(...).sort_values().plot(kind="bar")` is the bread-and-butter pipeline of category comparisons.
3. **`hue=` replaces the loop.** One argument splits any seaborn chart by category — colors, legend and all.
4. Match the **chart type** to the question: **line** for trends, **bar** for categories, **histogram/box** for distributions, **scatter** for relationships, **heatmap** for matrices.
5. Every pandas/seaborn call returns a matplotlib **`Axes`** — pass `ax=` to compose dashboards, call `ax.set_*` for anything the arguments don't cover.
6. Always set a **title** and axis labels; end plotting cells with `;`.
7. Save with `plt.savefig("name.png", dpi=200, bbox_inches="tight")` for reports.

## ✅ Self-assessment

- [ ] Plot a Series or DataFrame as a line chart in one statement
- [ ] Build a sorted bar chart from a `groupby` result
- [ ] Compare two distributions with `sns.histplot(..., hue=...)`
- [ ] Show a relationship with `sns.regplot` and read the trend line
- [ ] Draw a correlation heatmap with annotated cells
- [ ] Compose a 2×2 dashboard with `plt.subplots(2, 2)` and `ax=`
- [ ] Save a chart to PNG at publication DPI

## 🚀 Next step

Continue with **Notebook 10 — Statistics That Pay for Themselves**, where you'll learn to tell real differences from noise — the foundation for evaluating every model and A/B test that follows.